# nb_rotina_nexti
Rotinas de alta frequência da Nexti, do Bronze à Gold numa única sessão de notebook. Um só agendamento
(a cada 5 min) roda, em sequência, as rotinas de `ROTINAS` que estão ativas, dentro da janela de horário
e com o intervalo vencido. Para cada rotina:
1. `nb_ingestao_api` busca na API só as tabelas da rotina, desde o último watermark;
2. sem nenhuma alteração na API, a rotina termina aqui (só uma linha SEM_ALTERACAO no log);
3. com alteração, `nb_orquestrador` leva o lote para a Silver e as Golds recalculam só o que mudou.

Incluir uma rotina = uma entrada em `ROTINAS`. Rodar uma rotina sob demanda, fora do horário:
executar o notebook com `rotina = "AUSENCIAS"`.

A trava `Files/rotinas/nb_rotina_nexti.lock` (lakehouse Bronze) impede duas execuções simultâneas; a
seguinte termina sem fazer nada. A trava expira em `trava_minutos`, caso uma execução morra sem
liberá-la. As tabelas destas rotinas ficam fora do ciclo geral (TABELAS_ROTINA no nb_ingestao_api).

In [ ]:
rotina = ""                   # vazio = as rotinas no horário (agendamento); "AUSENCIAS" = só ela, a qualquer hora
modo_gold = "INCREMENTAL"     # COMPLETO recalcula as Golds inteiras; NENHUM pula as Golds (carga de histórico)
max_dias_por_execucao = 45    # janela máxima por tabela (recuperação de histórico)
configuracao_api = '{"NEXTI": {"key_vault": "https://kv-dw-fabric-kmf.vault.azure.net/"}}'
trava_minutos = 30
timeout_etapa_s = 3000

In [ ]:
import os, json, time, datetime as dt
from pyspark.sql import SparkSession, DataFrame, Window, functions as F, types as T
from delta.tables import DeltaTable

# Não renomear os lakehouses: o banco do SQL endpoint guarda o nome da criação e a sincronização falha
LH_BRONZE, LH_SILVER, LH_GOLD = "Bronze", "Silver", "Gold"
FUSO_BRT = dt.timezone(dt.timedelta(hours=-3))

# processo e sessão em UTC: as datas da Bronze são horário de Brasília rotulado UTC e não podem ser
# deslocadas na leitura, no CAST para DATE nem na volta para o Python (collect devolve hora local)
os.environ["TZ"] = "UTC"
time.tzset()
spark = SparkSession.builder.getOrCreate()   # no Fabric a sessão já existe
spark.conf.set("spark.sql.session.timeZone", "UTC")
# coluna nova na origem entra no MERGE e no append sem recriar a tabela
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

_LOCAL = os.environ.get("DW_LOCAL_ROOT")  # usado só em testes fora do Fabric
if _LOCAL:
    def caminho(lakehouse, tabela):
        return f"{_LOCAL}/{lakehouse}/{tabela}"
    def sair(valor):
        globals()["_EXIT_VALUE"] = valor
    spark.sparkContext.setCheckpointDir(f"{_LOCAL}/_checkpoints")
else:
    import notebookutils
    # IDs, e não nomes, no endereço do OneLake: o ABFS do Spark não aceita os espaços do nome do
    # workspace ("Squad Dados - DW Dev")
    _WS = notebookutils.runtime.context["currentWorkspaceId"]
    _LH = {}
    def raiz(lakehouse):
        """abfss://<workspace>@onelake.../<lakehouse>: + /Tables/<tabela> ou + /Files/<caminho>."""
        if lakehouse not in _LH:
            item = notebookutils.lakehouse.get(lakehouse)
            _LH[lakehouse] = item["id"] if isinstance(item, dict) else item.id
        return f"abfss://{_WS}@onelake.dfs.fabric.microsoft.com/{_LH[lakehouse]}"
    def caminho(lakehouse, tabela):
        return f"{raiz(lakehouse)}/Tables/{tabela}"
    def sair(valor):
        notebookutils.notebook.exit(valor)
    # resultados intermediários de materializar(), uma pasta por sessão (a rotina diária apaga as antigas)
    CHECKPOINTS = f"{raiz(LH_BRONZE)}/Files/_checkpoints"
    spark.sparkContext.setCheckpointDir(f"{CHECKPOINTS}/{spark.sparkContext.applicationId}")

def existe(lakehouse, tabela):
    try:
        return DeltaTable.isDeltaTable(spark, caminho(lakehouse, tabela))
    except Exception:
        return False

def ler(lakehouse, tabela, filtros=None, colunas=None):
    """DataFrame da tabela. filtros: [(coluna, operador, valor)], operadores =, >, >=, <, <=, in."""
    df = spark.read.format("delta").load(caminho(lakehouse, tabela))
    for coluna, op, valor in filtros or []:
        c = F.col(f"`{coluna}`")
        df = df.where(c.isin(list(valor)) if op == "in" else
                      {"=": c == valor, ">": c > valor, ">=": c >= valor, "<": c < valor, "<=": c <= valor}[op])
    return df.select(*[f"`{c}`" for c in colunas]) if colunas else df

def gravar(lakehouse, tabela, dados, modo="append"):
    """append acrescenta colunas novas (mergeSchema); overwrite troca dados e schema."""
    (dados.write.format("delta").mode(modo)
        .option("overwriteSchema" if modo == "overwrite" else "mergeSchema", "true")
        .save(caminho(lakehouse, tabela)))

def tabela_delta(lakehouse, tabela):
    return DeltaTable.forPath(spark, caminho(lakehouse, tabela))

def mesclar(lakehouse, tabela, dados, chaves):
    """MERGE pela chave: atualiza as linhas existentes e insere as novas (cria a tabela se não existir)."""
    if not existe(lakehouse, tabela):
        gravar(lakehouse, tabela, dados, modo="overwrite")
        return
    (tabela_delta(lakehouse, tabela).alias("t")
        .merge(dados.alias("s"), " AND ".join(f"t.`{c}` = s.`{c}`" for c in chaves))
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())

def apagar(lakehouse, tabela, coluna, valores, extra=None):
    """DELETE das linhas com `coluna` em `valores`, em blocos (listas grandes de ids)."""
    alvo = tabela_delta(lakehouse, tabela)
    valores = list(valores)
    for i in range(0, len(valores), 5000):
        cond = F.col(f"`{coluna}`").isin(valores[i:i + 5000])
        alvo.delete(cond & F.expr(extra) if extra else cond)

def linhas_deletadas(lakehouse, tabela):
    """Linhas apagadas pela última operação (o DELETE do Spark não devolve métricas)."""
    m = tabela_delta(lakehouse, tabela).history(1).select("operationMetrics").first()[0] or {}
    return int(m.get("numDeletedRows") or 0)

def criar(linhas, schema):
    """DataFrame a partir de dicts. schema em DDL, ex.: "id string, n bigint, dt timestamp"."""
    return spark.createDataFrame(linhas, schema=schema, verifySchema=False)

def _utc(v):
    return v.replace(tzinfo=dt.timezone.utc) if isinstance(v, dt.datetime) and v.tzinfo is None else v

def coletar(df):
    """Linhas como dicts, com timestamp em UTC com fuso (o collect devolve sem fuso, em UTC)."""
    return [{k: _utc(v) for k, v in r.asDict().items()} for r in df.collect()]

def materializar(df):
    """Executa já e grava o resultado em arquivo: necessário antes de apagar ou regravar a tabela que o
    originou. Em arquivo, e não na memória (localCheckpoint): um executor desligado no meio da execução
    levava o resultado junto (CHECKPOINT_RDD_BLOCK_ID_NOT_FOUND)."""
    return df.checkpoint(eager=True)

def nome_bronze(tabela_destino):
    """ctl_tabelas guarda o nome base (fato_contas_receber_gs); cada camada recebe o seu prefixo."""
    return f"bronze_{tabela_destino}"

def nome_silver(tabela_destino):
    return f"silver_{tabela_destino}"

def agora_utc():
    return dt.datetime.now(dt.timezone.utc)

def maximo(df, coluna):
    return _utc(df.agg(F.max(f"`{coluna}`")).first()[0])

def watermark_gold(gold, fonte):
    """Último _dt_carga da Silver já levado para esta Gold (tabela ctl_gold na Bronze). As Golds são
    incrementais por esta marca, e não pelo lote: se uma Gold falhar, a próxima execução retoma dali."""
    if not existe(LH_BRONZE, "ctl_gold"):
        return None
    r = ler(LH_BRONZE, "ctl_gold").where((F.col("gold") == gold) & (F.col("fonte") == fonte)).first()
    return _utc(r["ultimo_dt_carga"]) if r else None

def gravar_watermark_gold(gold, fonte, valor):
    linha = criar([{"gold": gold, "fonte": fonte, "ultimo_dt_carga": valor, "dt_atualizacao": agora_utc()}],
                  "gold string, fonte string, ultimo_dt_carga timestamp, dt_atualizacao timestamp")
    com_nova_tentativa(lambda: mesclar(LH_BRONZE, "ctl_gold", linha, ["gold", "fonte"]))

def com_nova_tentativa(acao, tentativas=5):
    """Execuções diferentes podem gravar a mesma tabela de controle ao mesmo tempo: o commit que perde a
    disputa é refeito sobre a versão nova da tabela."""
    for n in range(tentativas):
        try:
            return acao()
        except Exception as e:
            msg = str(e).lower()
            if n == tentativas - 1 or not any(p in msg for p in ("concurrent", "conflict", "commit")):
                raise
            time.sleep(5 * (n + 1))

def consulta(nome):
    """Texto de uma consulta da pasta `consultas` (no Fabric: lakehouse Gold → Files/consultas/<nome>.sql,
    copiada de implantacao/consultas pela implantação). Nos testes locais, lê de DW_CONSULTAS."""
    if _LOCAL:
        with open(os.path.join(os.environ["DW_CONSULTAS"], f"{nome}.sql"), encoding="utf-8") as arq:
            return arq.read()
    return notebookutils.fs.head(
        f"{raiz(LH_GOLD)}/Files/consultas/{nome}.sql", 10_000_000)

def cabecalho(sql):
    """Configuração no topo do .sql, em linhas `-- @chave valor` (ex.: `-- @destino gold_fato_coberturas`)."""
    cfg = {}
    for linha in sql.splitlines():
        linha = linha.strip()
        if linha.startswith("-- @"):
            chave, _, valor = linha[4:].partition(" ")
            cfg[chave] = valor.strip()
    return cfg

def ler_config(nome):
    """Arquivo YAML da pasta config (no Fabric: lakehouse Gold → Files/config/<nome>, copiado de config/ pela
    implantação). Nos testes locais, lê de DW_CONFIG."""
    import yaml
    if _LOCAL:
        with open(os.path.join(os.environ["DW_CONFIG"], nome), encoding="utf-8") as arq:
            return yaml.safe_load(arq)
    return yaml.safe_load(notebookutils.fs.head(
        f"{raiz(LH_GOLD)}/Files/config/{nome}", 10_000_000))

# ---------------------------------------------------------------- aviso no Teams por execução
# Resultado de cada ciclo (nb_orquestrador) e de cada rotina (nb_rotina_nexti) vai para o Teams pelo
# mesmo Workflow do nb_monitoramento. Config: config/qualidade.yml, seção `monitoramento.notificacao`.
_ICONE_STATUS = {"SUCESSO": "✅", "PARCIAL": "⚠️", "FALHA": "❌"}
_ICONE_COMP = {"NORMAL": "🟢", "ATENÇÃO": "🟡", "ALERTA": "🔴"}
_COR_COMP = {"NORMAL": "Good", "ATENÇÃO": "Warning", "ALERTA": "Attention"}

def _config_notificacao():
    return ((ler_config("qualidade.yml").get("monitoramento") or {}).get("notificacao")) or {}

def incrementar_contador(chave):
    """Contador persistente por chave (ctl_contador na Bronze); devolve o novo valor. Usado para avisar
    só 1 a cada N execuções das rotinas de alta frequência."""
    atual = 0
    if existe(LH_BRONZE, "ctl_contador"):
        r = ler(LH_BRONZE, "ctl_contador").where(F.col("chave") == chave).first()
        atual = int(r["n"]) if r else 0
    linha = criar([{"chave": chave, "n": atual + 1, "dt": agora_utc()}], "chave string, n bigint, dt timestamp")
    com_nova_tentativa(lambda: mesclar(LH_BRONZE, "ctl_contador", linha, ["chave"]))
    return atual + 1

def enviar_teams(corpo_cartao):
    """Posta um Adaptive Card no Workflow do Teams (URL no Key Vault, cofre e segredo em qualidade.yml).
    Não lança: uma notificação que falha não derruba a execução. Devolve o status do envio em texto."""
    notif = _config_notificacao()
    if _LOCAL:
        return "local"
    if not notif.get("teams_workflow_segredo") or not notif.get("key_vault"):
        return "sem workflow"
    try:
        import requests
        url = notebookutils.credentials.getSecret(notif["key_vault"], notif["teams_workflow_segredo"])
        payload = {"type": "message", "destinatario": notif.get("destinatario"),
                   "attachments": [{"contentType": "application/vnd.microsoft.card.adaptive", "content": {
                       "$schema": "http://adaptivecards.io/schemas/adaptive-card.json", "type": "AdaptiveCard",
                       "version": "1.4", "body": corpo_cartao}}]}
        r = requests.post(url, json=payload, timeout=30)
        r.raise_for_status()
        return f"enviado para {notif.get('destinatario') or 'o Workflow'}"
    except Exception as e:   # sem Workflow configurado ou fora do ar: a execução segue normal
        return f"não notificado: {str(e)[:200]}"

def comportamento_execucao(status, duracao_s, limite_atencao=None):
    """🔴 ALERTA: a base não atualizou (FALHA/PARCIAL). 🟡 ATENÇÃO: concluiu, mas acima do tempo esperado
    (`limite_atencao`). 🟢 NORMAL: concluiu no tempo normal."""
    if status in ("FALHA", "PARCIAL"):
        return "ALERTA"
    if limite_atencao and duracao_s and duracao_s > limite_atencao:
        return "ATENÇÃO"
    return "NORMAL"

def avisar_execucao(base, objeto, status, duracao_s, erro="", resumo="", tipo="CICLO", chave=None):
    """Aviso no Teams do resultado de uma execução (ciclo do pipeline ou rotina). As rotinas da Nexti só
    avisam o sucesso NORMAL 1 vez a cada `nexti_cada` execuções (config); falha, PARCIAL e ATENÇÃO avisam
    sempre. Devolve o status do envio ('enviado...', 'local', 'agrupado', 'desligado', 'sem workflow')."""
    notif = _config_notificacao()
    if not notif.get("avisar_execucoes", True):
        return "desligado"
    comp = comportamento_execucao(status, duracao_s, (notif.get("duracao_atencao_s") or {}).get(tipo))
    cada = int(notif.get("nexti_cada", 1)) if str(base).upper() == "NEXTI" else 1
    if comp == "NORMAL" and cada > 1 and chave and incrementar_contador(f"AVISO_{chave}") % cada != 1:
        return "agrupado"
    fatos = [("BASE", base), ("Notebook/Pipeline", objeto),
             ("Status", f"{_ICONE_STATUS.get(status, '•')} {status}")]
    if erro:
        fatos.append(("Erro", erro[:800]))
    fatos.append(("Horário", agora_utc().astimezone(FUSO_BRT).strftime("%d/%m/%Y %H:%M")))
    fatos.append(("Comportamento", f"{_ICONE_COMP[comp]} {comp}"))
    corpo = [{"type": "TextBlock", "size": "Medium", "weight": "Bolder", "color": _COR_COMP[comp],
              "text": f"{_ICONE_COMP[comp]} DW Fabric — {comp}"}]
    if resumo:
        corpo.append({"type": "TextBlock", "isSubtle": True, "wrap": True, "spacing": "None", "text": resumo[:300]})
    corpo.append({"type": "FactSet", "facts": [{"title": t, "value": str(v)} for t, v in fatos]})
    return enviar_teams(corpo)

In [ ]:
import uuid

# tabelas: endpoints da Nexti (ENDPOINTS_NEXTI no nb_ingestao_api)
# golds: em ordem; "nb_..." = notebook próprio, outro nome = consulta consultas/<nome>.sql executada pelo nb_gold
# intervalo_min: de quanto em quanto tempo roda; janela: horário de Brasília em que roda (inclusive)
ROTINAS = {
    "AUSENCIAS": {
        "tabelas": ["WORKPLACETRANSFERS", "ABSENCES"],
        "golds": ["nb_gold_fato_movimentacao_colaborador", "nb_gold_fato_ausencias"],
        "intervalo_min": 5, "janela": ("06:00", "20:00"), "ativo": True,
    },
    "COBERTURAS": {
        "tabelas": ["REPLACEMENTS"],
        "golds": ["gold_fato_coberturas"],
        "intervalo_min": 5, "janela": ("06:00", "20:00"), "ativo": True,
    },
    "MARCACAO": {
        "tabelas": ["CLOCKINGS"],
        "golds": ["gold_fato_marcacao_ponto"],
        "intervalo_min": 5, "janela": ("06:00", "20:00"), "ativo": False,  # ativar após a carga do histórico
    },
    # cadastros de que as Golds dependem: só sob demanda (o ciclo geral COMPLETO os atualiza às 05:00)
    "CADASTROS": {
        "tabelas": ["PERSONS", "WORKPLACES", "COMPANIES", "USERACCOUNTS"],
        "golds": [],
        "intervalo_min": None, "janela": None, "ativo": False,
    },
}
inicio = agora_utc()
_log = []

def log(lote, rot, tabela_id, etapa, linhas, status, msg=""):
    _log.append({"lote": lote, "ciclo": f"ROTINA_{rot}", "tabela_id": tabela_id, "etapa": etapa,
                 "linhas": int(linhas or 0), "status": status, "mensagem": (msg or "")[:4000],
                 "dt_registro": agora_utc()})

def gravar_log():
    if _log:
        com_nova_tentativa(lambda: gravar(LH_BRONZE, "ctl_log", criar(_log, (
            "lote string, ciclo string, tabela_id string, etapa string, linhas bigint, status string, "
            "mensagem string, dt_registro timestamp"))))
        _log.clear()

def no_horario(nome, cfg):
    """Ativa, dentro da janela e com o intervalo vencido desde a última execução."""
    if not cfg["ativo"] or not cfg["intervalo_min"]:
        return False
    agora = dt.datetime.now(FUSO_BRT)
    ini, fim = cfg["janela"]
    if not (ini <= agora.strftime("%H:%M") <= fim):
        return False
    ultima = watermark_gold(f"ROTINA_{nome}", "execucao")
    # 1 min de folga: o agendamento não dispara exatamente no mesmo segundo
    return ultima is None or agora_utc() - ultima >= dt.timedelta(minutes=cfg["intervalo_min"] - 1)

if rotina:
    if rotina not in ROTINAS:
        raise ValueError(f"Rotina desconhecida: {rotina}. Disponíveis: {', '.join(ROTINAS)}")
    escolhidas = [rotina]
else:
    escolhidas = [n for n, c in ROTINAS.items() if no_horario(n, c)]

In [ ]:
if _LOCAL:
    def pegar_trava():
        return True
    def soltar_trava():
        pass
    def rodar(nome, params):
        raise RuntimeError("nb_rotina_nexti só roda no Fabric (chama outros notebooks)")
else:
    TRAVA = f"{raiz(LH_BRONZE)}/Files/rotinas/nb_rotina_nexti.lock"
    DONO = str(uuid.uuid4())

    def pegar_trava():
        if notebookutils.fs.exists(TRAVA):
            try:
                desde = dt.datetime.fromisoformat(notebookutils.fs.head(TRAVA, 100).split("|")[1])
            except Exception:
                desde = agora_utc()
            if agora_utc() - desde < dt.timedelta(minutes=int(trava_minutos)):
                return False
            print(f"trava vencida desde {desde:%H:%M}: assumindo a rotina")
        notebookutils.fs.put(TRAVA, f"{DONO}|{agora_utc().isoformat()}", True)
        return True

    def soltar_trava():
        try:
            if notebookutils.fs.head(TRAVA, 200).startswith(DONO):
                notebookutils.fs.rm(TRAVA)
        except Exception:
            pass

    def rodar(nome, params):
        return notebookutils.notebook.run(nome, int(timeout_etapa_s), params)

def manutencao(nome, cfg, lote):
    """Uma vez por dia: compacta os arquivos pequenos que a rotina gera a cada 5 min e remove os
    antigos (VACUUM de 7 dias). Falha aqui só vai para o log."""
    alvos = [(LH_BRONZE, "ctl_log"), (LH_BRONZE, "ctl_tabelas"), (LH_BRONZE, "ctl_gold")]
    for t in coletar(ler(LH_BRONZE, "ctl_tabelas")):
        if t["origem"] == "NEXTI" and t["tabela_origem"] in cfg["tabelas"]:
            alvos += [(LH_BRONZE, nome_bronze(t["tabela_destino"])), (LH_SILVER, nome_silver(t["tabela_destino"]))]
    alvos += [(LH_GOLD, g.replace("nb_", "", 1)) for g in cfg["golds"]]
    for lh, tabela in alvos:
        try:
            if existe(lh, tabela):
                alvo = tabela_delta(lh, tabela)
                alvo.optimize().executeCompaction()
                alvo.vacuum(168)
        except Exception as e:
            log(lote, nome, tabela, "MANUTENCAO", 0, "FALHA", str(e))
    # checkpoints de sessões encerradas (mais de 1 dia): os da sessão atual e de outras em andamento ficam
    if not _LOCAL:
        try:
            limite_ms = (agora_utc() - dt.timedelta(days=1)).timestamp() * 1000
            for pasta in notebookutils.fs.ls(CHECKPOINTS):
                if pasta.modifyTime < limite_ms:
                    notebookutils.fs.rm(pasta.path, True)
        except Exception as e:
            log(lote, nome, "_checkpoints", "MANUTENCAO", 0, "FALHA", str(e))
    # regras de qualidade do grupo da rotina (config/qualidade.yml), uma vez por dia
    try:
        q = json.loads(rodar("nb_qualidade", {"grupo": nome, "lote": lote}) or "{}")
        status = "SUCESSO" if not q.get("falhas_erro") else "FALHA"
        log(lote, nome, f"QUALIDADE.{nome}", "QUALIDADE", q.get("regras", 0), status,
            json.dumps(q, ensure_ascii=False))
    except Exception as e:
        log(lote, nome, f"QUALIDADE.{nome}", "QUALIDADE", 0, "FALHA", str(e))
    gravar_watermark_gold(f"ROTINA_{nome}", "manutencao", agora_utc())

def hoje_ja(nome, marca):
    ultima = watermark_gold(f"ROTINA_{nome}", marca)
    return ultima is not None and ultima.astimezone(FUSO_BRT).date() == dt.datetime.now(FUSO_BRT).date()

def executar(nome, cfg):
    lote = f"rotina-{nome.lower()}-{uuid.uuid4()}"
    t_ini = agora_utc()
    gravar_watermark_gold(f"ROTINA_{nome}", "execucao", t_ini)
    res_api = rodar("nb_ingestao_api", {
        "lote": lote, "ciclo": "FATOS", "origens": "NEXTI", "tabelas": ",".join(cfg["tabelas"]),
        "configuracao_api": configuracao_api, "max_dias_por_execucao": int(max_dias_por_execucao)})
    itens_api = json.loads(res_api or "[]")
    copiadas = sum(int(float(s.split("|")[2] or 0)) for s in itens_api)
    falhou = any(s.split("|")[1] == "FALHA" for s in itens_api)
    # nada mudou na API e as Golds já viraram o dia: termina sem Silver nem Gold. Os watermarks não
    # avançam; a próxima busca parte do mesmo ponto, o que só alarga um pouco a janela consultada.
    if copiadas == 0 and not falhou and modo_gold != "COMPLETO" and (not cfg["golds"] or hoje_ja(nome, "golds_dia")):
        seg = (agora_utc() - t_ini).total_seconds()
        log(lote, nome, f"ROTINA.{nome}", "ROTINA", 0, "SEM_ALTERACAO", f"nenhuma alteração na API | {seg:.0f}s")
        return {"status": "SUCESSO", "resumo": "sem alteração na API", "segundos": seg,
                "linha": f"{nome}: sem alteração ({seg:.0f}s)"}
    rodar("nb_orquestrador", {
        "lote": lote, "ciclo": "FATOS", "resultados": "[]", "resultados_api": res_api,
        "status_ciclo": "SUCESSO", "origens": "NEXTI", "notificar": "NAO"})   # a rotina faz o próprio aviso
    resumo = [f"{copiadas} linhas da API"]
    for g in (cfg["golds"] if modo_gold != "NENHUM" else []):
        t0 = agora_utc()
        if g.startswith("nb_"):
            r = json.loads(rodar(g, {"lote": lote, "modo": modo_gold}) or "{}")
        else:
            r = json.loads(rodar("nb_gold", {"gold": g, "lote": lote, "modo": modo_gold}) or "{}")
        log(lote, nome, r.get("gold", g), "GOLD", r.get("linhas", 0), "SUCESSO",
            f"{json.dumps(r, ensure_ascii=False)} | {(agora_utc() - t0).total_seconds():.0f}s")
        resumo.append(f"{r.get('gold', g)}: {r.get('linhas', 0)} linhas")
    if cfg["golds"] and modo_gold != "NENHUM":
        gravar_watermark_gold(f"ROTINA_{nome}", "golds_dia", agora_utc())
    if not hoje_ja(nome, "manutencao"):
        manutencao(nome, cfg, lote)
        resumo.append("manutenção diária")
    seg = (agora_utc() - t_ini).total_seconds()
    log(lote, nome, f"ROTINA.{nome}", "ROTINA", copiadas, "SUCESSO", " | ".join(resumo) + f" | {seg:.0f}s")
    return {"status": "SUCESSO", "resumo": " | ".join(resumo), "segundos": seg,
            "linha": f"{nome}: " + " | ".join(resumo) + f" | {seg:.0f}s"}

In [ ]:
resultado = {"rotinas": escolhidas, "status": "SEM_ROTINA_NO_HORARIO" if not escolhidas else "EM_EXECUCAO"}
falhas = []
if escolhidas and not pegar_trava():
    print("nb_rotina_nexti já em execução: nada a fazer")
elif escolhidas:
    try:
        for nome in escolhidas:
            t0 = agora_utc()
            try:
                r = executar(nome, ROTINAS[nome])
                print(r["linha"])
                print("  teams:", avisar_execucao("NEXTI", f"Rotina {nome}", r["status"], r["segundos"],
                                                  resumo=r["resumo"], tipo="ROTINA", chave=f"ROTINA_{nome}"))
            except Exception as e:
                falhas.append(f"{nome}: {e}")
                log("", nome, f"ROTINA.{nome}", "ROTINA", 0, "FALHA", str(e))
                avisar_execucao("NEXTI", f"Rotina {nome}", "FALHA", (agora_utc() - t0).total_seconds(),
                                erro=str(e), tipo="ROTINA", chave=f"ROTINA_{nome}")
            gravar_log()
        resultado = {"rotinas": escolhidas, "status": "FALHA" if falhas else "SUCESSO",
                     "segundos": round((agora_utc() - inicio).total_seconds())}
    finally:
        soltar_trava()
else:
    print("nenhuma rotina no horário")

In [ ]:
# monitoramento a cada 30 min (rotinas atrasadas, tabelas paradas, falhas, qualidade) -> ctl_alertas
ultimo_monitor = watermark_gold("MONITORAMENTO", "ctl_log")
if not _LOCAL and (ultimo_monitor is None or agora_utc() - ultimo_monitor > dt.timedelta(minutes=30)):
    try:
        print("monitoramento:", rodar("nb_monitoramento", {}))
    except Exception as e:
        print(f"monitoramento falhou: {str(e)[:300]}")

In [ ]:
# falha de uma rotina não impede as outras; no fim, a execução fica marcada como falha no Fabric
if falhas:
    raise RuntimeError("Rotinas com falha:\n" + "\n".join(falhas))
# fora do try: no Fabric o exit pode ser tratado como exceção e cair no except
sair(json.dumps(resultado))